<a href="https://colab.research.google.com/github/Pinkraaaa/Lab_3/blob/main/Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PART 0 AND 1.1
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)   # use this everywhere so your results are reproducible
np.random.seed(RANDOM_STATE)

print("Running on:", DEVICE)
print()


#PART 1.1
# TODO: Generate the toy data
#   x = np.linspace(-3, 3, 100); y = np.sin(x) + small Gaussian noise (std=0.1)
#   Plot it.

np.random.seed(RANDOM_STATE)
x = np.linspace(-3, 3, 100)
y = np.sin(x) + np.random.normal(0, 0.1, size = x.shape)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, s = 10, label = "noisy samples")
plt.plot(x, np.sin(x), color = "blue", lw = 1, label = "true sin(x)")
plt.title("Toy data: y = sin(x) + noise")
plt.legend()
plt.tight_layout()
plt.show()


# TODO: Initialise w and b to small random values.

rng = np.random.default_rng(RANDOM_STATE)
w = rng.normal(0, 0.1)
b = rng.normal(0, 0.1)

# TODO: Implement ONE gradient-descent step by hand, then loop it for ~200 steps:
#   y_hat = w * x + b
#   loss  = np.mean((y_hat - y) ** 2)               # MSE
#   dw    = np.mean(2 * (y_hat - y) * x)            # dLoss/dw  (derive this on paper!)
#   db    = np.mean(2 * (y_hat - y))                # dLoss/db
#   w    -= lr * dw
#   b    -= lr * db
#   Record the loss at every step.

# TODO: Plot (a) the loss curve over the 200 steps
lr = 0.1
n_steps = 200
losses = []

for step in range(n_steps):
    y_hat = w * x + b
    loss = np.mean((y_hat - y) ** 2)          #MSE
    losses.append(loss)

    dw = np.mean(2 * (y_hat - y) * x)         #dLoss/dw
    db = np.mean(2 * (y_hat - y))             #dLoss/db

    w -= lr * dw
    b -= lr * db

print()
print(f"Final params: w = {w:.2f}, b = {b:.2f}, final loss = {losses[-1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize = (12, 6))
axes[0].plot(losses)
axes[0].set_title("Loss curve (single neuron)")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE loss")
print()



#(b) the data with the fitted line on top.
axes[1].scatter(x, y, s = 10, label = "data")
axes[1].plot(x, w * x + b, color = "pink", lw = 2, label = f"fit: y = {w:.2f}x + {b:.2f}")
axes[1].set_title("Fitted line vs data")
axes[1].legend()
plt.tight_layout()
plt.show()




PART 1.2: HIDDEN LAYERS AND ACTIVATION: WHY DEPTHS HELPS

In [ ]:
#PART 1.2
# TODO: Initialise parameters
#   W1: shape (1, 8),  b1: shape (8,)
#   W2: shape (8, 1),  b2: shape (1,)
#   Small random values (e.g. np.random.randn(...) * 0.5)
np.random.seed(RANDOM_STATE)
W1 = np.random.randn(1, 8) * 0.5
b1 = np.zeros(8)
W2 = np.random.randn(8, 1) * 0.5
b2 = np.zeros(1)

x_col = x.reshape(-1, 1)   # (100, 1) so matrix multiply works
y_col = y.reshape(-1, 1)   # (100, 1)

lr = 0.05
steps = 3000
losses = []


# TODO: Forward pass (keep the intermediate values — you need them for backprop):
#   z1 = x @ W1 + b1        # pre-activation, shape (100, 8)
#   h  = np.tanh(z1)        # activation
#   y_hat = h @ W2 + b2     # output, shape (100, 1)
#   loss  = np.mean((y_hat - y) ** 2)

for step in range(steps):
    # ---- forward pass ----
    z1 = x_col @ W1 + b1        # (100, 8)
    h = np.tanh(z1)              # (100, 8)
    y_hat = h @ W2 + b2          # (100, 1)

    loss = np.mean((y_hat - y_col) ** 2)
    losses.append(loss)

# TODO: Backward pass (the chain rule, layer by layer):
#   d_yhat = 2 * (y_hat - y) / len(y)
#   dW2 = h.T @ d_yhat ;  db2 = d_yhat.sum(axis=0)
#   dh  = d_yhat @ W2.T
#   dz1 = dh * (1 - np.tanh(z1) ** 2)     # derivative of tanh
#   dW1 = x.T @ dz1 ;  db1 = dz1.sum(axis=0)

# ---- backward pass ----
    d_yhat = 2 * (y_hat - y_col) / len(y_col)   # (100, 1)
    dW2 = h.T @ d_yhat                           # (8, 1)
    db2 = d_yhat.sum(axis=0)                     # (1,)

    dh = d_yhat @ W2.T                           # (100, 8)
    dz1 = dh * (1 - np.tanh(z1) ** 2)            # (100, 8)
    dW1 = x_col.T @ dz1                          # (1, 8)
    db1 = dz1.sum(axis=0)

# TODO: Update all parameters with learning rate lr, loop for ~3000 steps,
#   record the loss.
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2


# TODO: Plot the loss curve AND the final fit over the data.
#   Then re-run with the tanh REMOVED (h = z1). What happens to the fit?

plt.figure(figsize = (6, 4))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training loss (hidden layer, tanh)")
plt.show()


plt.figure(figsize = (6, 4))
plt.scatter(x, y, s = 10, label = "data", alpha = 0.5)
plt.plot(x, y_hat, color = "green", label = "fitted network")
plt.legend()
plt.title("Fit with hidden layer + tanh")
plt.show()



